In [1]:
# Setup
import os
import sys
from pathlib import Path

NOTEBOOK_ROOT = Path("/scratch/jq2uw/MME/instruct_vlm_edit")
os.chdir(NOTEBOOK_ROOT)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))

from revlm import *
import argparse
import pandas as pd
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

from revlm.metrics.utils.r_gen import get_r_gen_input

In [ ]:
# Config
dataset_name = "aokvqa"  # or "aokvqa"


In [3]:
# (1) Load union of edit_ds uids from all 4 VLMs
import json

MODEL_TAGS = [
    "llava-1.5-7b-hf",
    "instructblip-vicuna-7b",
    "Qwen3-VL-8B-Instruct",
    "Qwen3-VL-4B-Instruct",
]

edit_uids_all = set()
for model_tag in MODEL_TAGS:
    pred_path = f"results/pred/{model_tag}/{dataset_name}/mc_all.json"
    if not os.path.exists(pred_path):
        print(f"  Skip (not found): {pred_path}")
        continue
    with open(pred_path, "r") as f:
        preds = json.load(f)
    # Filter to errors only: gold['label'] != pred['label_maxprob']
    errors = [p for p in preds if p['gold']['label'] != p['pred']['label_maxprob']]
    uids = {str(p["uid"]) for p in errors}
    edit_uids_all.update(uids)
    print(f"  {model_tag}: {len(errors)} errors / {len(preds)} total")

print(f"\nUnion of edit uids across all VLMs: {len(edit_uids_all)}")

  llava-1.5-7b-hf: 7197 errors / 18195 total


  instructblip-vicuna-7b: 5954 errors / 18195 total
  Qwen3-VL-8B-Instruct: 7139 errors / 18195 total
  Qwen3-VL-4B-Instruct: 5188 errors / 18195 total

Union of edit uids across all VLMs: 11126


In [4]:
# (1) Load full dataset to get uid -> original_image mapping
args = argparse.Namespace(split="all", dataset_name=dataset_name)
config = configure_args(args, config_path=None)
ds = VQADataset(config)
df_full = ds.load_df()

# Build uid -> original image path mapping
uid_to_original_image = dict(zip(df_full["uid"].astype(str), df_full["image_path"]))
print(f"Loaded {len(uid_to_original_image)} uid->image mappings for {dataset_name}")


Task evaluation metrics will be saved to results/te/ft/llava-1.5-7b-hf/aokvqa
Edit evaluation metrics will be saved to results/ee/ft/llava-1.5-7b-hf/aokvqa
Predictions will be saved to results/pred/llava-1.5-7b-hf/aokvqa
Post-edit predictions will be saved to results/pred_postedit/ft/llava-1.5-7b-hf/aokvqa
Unified filename to save: mc_all.json


Loaded 18195 uid->image mappings for aokvqa


In [5]:
# (3) Load r_gen_df, filter to edit_uids_all only
r_gen_df = get_r_gen_input(dataset_name, edit_ds=None, s=0)
print(f"Loaded r_gen_df: {len(r_gen_df)} rows (full)")

# Filter to only uids in edit_uids_all
r_gen_df = r_gen_df[r_gen_df["uid"].astype(str).isin(edit_uids_all)].reset_index(drop=True)
print(f"After filtering to edit_uids_all: {len(r_gen_df)} rows")
r_gen_df.head()


Loaded r_gen_df: 132119 rows (full)
After filtering to edit_uids_all: 80827 rows


,uid,sid,question,answer,rationale,choices,idx_choices,image_path
0,1,1_1,What is next to the man in the image?,Some bags,The image shows a man standing by some bags on...,Some bags; A dog; A bicycle; A tree,(A) Some bags\n(B) A dog\n(C) A bicycle\n(D) A...,data/r_gen/image/aokvqa/1_1.png
1,1,1_2,What is the man standing next to?,Bags on street,The image shows a man standing by some bags on...,Bags on street; Train on street; Tree on stree...,(A) Bags on street\n(B) Train on street\n(C) T...,data/r_gen/image/aokvqa/1_2.png
2,1,1_3,What is the man likely not doing?,Waiting for delivery,The image shows a man standing by some bags on...,Waiting for delivery; Standing by a train; Sit...,(A) Waiting for delivery\n(B) Standing by a tr...,data/r_gen/image/aokvqa/1_3.png
3,1,1_4,What is the skateboarder doing?,Not paying attention,The image shows a man standing by some bags on...,Not paying attention; Delivering luggage; Stan...,(A) Not paying attention\n(B) Delivering lugga...,data/r_gen/image/aokvqa/1_4.png
4,1,1_5,Where would a train not likely be found?,On the street,A train would not be on the street.,On the street; In a station; On the tracks; In...,(A) On the street\n(B) In a station\n(C) On th...,data/r_gen/image/aokvqa/1_5.png


In [6]:
# (3) Load QWEN3-8B model
model_name = "qwen3" 

args = argparse.Namespace(
    config="revlm/config/config.yaml",
    split="all", 
    dataset_name=dataset_name, 
    model_name=model_name,
)
config = configure_args(args, config_path=args.config)
model = VQAModel(config)
print(f"Loaded model: {model_name}")


Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/aokvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/aokvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/aokvqa
Post-edit predictions will be saved to results/pred_postedit/ft/Qwen3-VL-8B-Instruct/aokvqa
Unified filename to save: mc_all.json


Loaded model: qwen3


In [ ]:
# (4) VQA verification function
@torch.no_grad()
def get_yes_prob(model, image_path, rationale):
    """Ask model: does image show '{rationale}'? Return P(yes)."""
    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"Failed to load {image_path}: {e}")
        return 0.0  # Treat as bad image
    
    # Build VQA question from rationale
    question = f"Does this image show: {rationale.lower().rstrip('.')}?"
    
    # Get NLL for "Yes" and "No"
    nll_yes, _, _ = model.get_loss_y(img, question, "Yes")
    nll_no, _, _ = model.get_loss_y(img, question, "No")
    
    # Convert to probability: P(yes) = 1 / (1 + exp(nll_yes - nll_no))
    p_yes = 1.0 / (1.0 + np.exp(nll_yes - nll_no))
    return p_yes


In [ ]:
# (6) Process each row with checkpointing
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import json

# Checkpoint paths
ckpt_path = f"data/r_gen/remove/{dataset_name}_progress.json"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

# Load checkpoint if exists
if os.path.exists(ckpt_path):
    with open(ckpt_path, "r") as f:
        ckpt = json.load(f)
    bad_sids = ckpt["bad_sids"]
    processed_sids = set(ckpt["processed_sids"])
    print(f"Resumed from checkpoint: {len(processed_sids)} processed, {len(bad_sids)} bad")
else:
    bad_sids = []
    processed_sids = set()

results = []
low_p_count = len([s for s in bad_sids])  # Approximate
missing_count = 0
save_every = 100

for i, (idx, row) in enumerate(r_gen_df.iterrows()):
    sid = str(row["sid"])
    if sid in processed_sids:
        continue  # Skip already processed
    
    uid = str(row["uid"])
    gen_image_path = row["image_path"]
    rationale = row["rationale"]
    
    # Check if generated image exists
    if not os.path.exists(gen_image_path):
        missing_count += 1
        bad_sids.append(sid)
        is_bad = True
        status = "MISSING"
        p_yes = None
    else:
        p_yes = get_yes_prob(model, gen_image_path, rationale)
        if p_yes < 0.1:
            low_p_count += 1
            bad_sids.append(sid)
            is_bad = True
            status = f"BAD (P={p_yes:.2f})"
        else:
            is_bad = False
            status = f"KEEP (P={p_yes:.2f})"
    
    processed_sids.add(sid)
    results.append({"sid": sid, "uid": uid, "p_yes": p_yes, "is_bad": is_bad})
    
    # Save checkpoint every N processed
    if len(processed_sids) % save_every == 0:
        with open(ckpt_path, "w") as f:
            json.dump({"bad_sids": bad_sids, "processed_sids": list(processed_sids)}, f)
        clear_output(wait=True)
        print(f"[{len(processed_sids)}/{len(r_gen_df)}] Bad: {len(bad_sids)} | Last: {status}")
    
    if i % 100 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

# Final save
with open(ckpt_path, "w") as f:
    json.dump({"bad_sids": bad_sids, "processed_sids": list(processed_sids)}, f)

print(f"\n{'='*50}")
print(f"Done! Processed {len(processed_sids)} images")
print(f"Bad images to remove: {len(bad_sids)}")
print(f"Keep rate: {100*(len(r_gen_df)-len(bad_sids))/len(r_gen_df):.1f}%")


[37300/80827] Bad: 5904 | Last: KEEP (P=0.90)


In [ ]:
# (7) Summary of bad sids
print(f"Total processed: {len(processed_sids)}")
print(f"Bad sids to remove: {len(bad_sids)}")
print(f"Sample bad sids: {bad_sids[:10]}")

if len(results) > 0:
    results_df = pd.DataFrame(results)
    print(f"\nP(yes) distribution (this session):")
    print(results_df["p_yes"].dropna().describe())


In [ ]:
# (8) Save final bad sids to JSON (from checkpoint)
out_path = f"data/r_gen/remove/{dataset_name}.json"

with open(out_path, "w") as f:
    json.dump(bad_sids, f)

print(f"Saved {len(bad_sids)} bad sids to {out_path}")
print(f"You can now delete the checkpoint: {ckpt_path}")

In [ ]:
# (9) Visualize some bad examples (to be removed)
import matplotlib.pyplot as plt

if len(results) > 0:
    results_df = pd.DataFrame(results)
    bad_rows = results_df[results_df["is_bad"] & results_df["p_yes"].notna()].head(5)
    
    if len(bad_rows) > 0:
        fig, axes = plt.subplots(len(bad_rows), 2, figsize=(10, 4*len(bad_rows)))
        if len(bad_rows) == 1:
            axes = [axes]
        
        for i, (_, res_row) in enumerate(bad_rows.iterrows()):
            sid = res_row["sid"]
            uid = res_row["uid"]
            # Find row in r_gen_df by sid
            match = r_gen_df[r_gen_df["sid"].astype(str) == sid]
            if match.empty:
                continue
            orig_row = match.iloc[0]
            gen_path = orig_row["image_path"]
            orig_path = uid_to_original_image.get(uid)
            
            if os.path.exists(gen_path):
                axes[i][0].imshow(Image.open(gen_path))
            axes[i][0].set_title(f"BAD (P={res_row['p_yes']:.2f})\nsid={sid}")
            axes[i][0].axis("off")
            
            if orig_path and os.path.exists(orig_path):
                axes[i][1].imshow(Image.open(orig_path))
            axes[i][1].set_title(f"Original\n{orig_row['rationale'][:40]}...")
            axes[i][1].axis("off")
        
        plt.tight_layout()
        plt.show()
    else:
        print("No bad images with P(yes) values to visualize.")
else:
    print("No results yet.")
